In [1]:
import os

In [2]:

import json
from IPython.display import Audio, display
import openai
from gtts import gTTS
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
openai.api_key = os.getenv('OPENAI_API_KEY')

In [4]:
# 1) STT: transcribe Urdu audio using OpenAI Whisper model
def transcribe_audio(audio_path, model_preference=('whisper-1',)):
    """Transcribe audio file to Urdu script. Returns transcribed text."""
    with open(audio_path, 'rb') as af:
        resp = openai.audio.transcriptions.create(model=model_preference[0], file=af)
    return resp.text

In [5]:
# 2) Convert Urdu-script to Roman-Urdu
def urdu_to_roman(urdu_text, model='gpt-4o-mini', temperature=0.0):
    sys_msg = 'You are a helpful assistant that converts Urdu (Arabic script) into Roman-Urdu (Latin characters). Preserve meaning, punctuation and numbers.'
    prompt = f'Convert the following Urdu text into Roman Urdu (use natural conversational transliteration).\n\nText:\n{urdu_text}\n\nReturn ONLY the Roman-Urdu text.'
    
    resp = openai.chat.completions.create(
        model=model,
        messages=[{'role': 'system', 'content': sys_msg}, {'role': 'user', 'content': prompt}],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()

In [6]:
# 3) RAG: query the Chroma DB in ./chroma_db and generate a bilingual response
def rag_query_and_generate(roman_query, chroma_persist_dir='chroma_db', collection_name='sample', model='gpt-4o-mini'):
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import Chroma

    emb = OpenAIEmbeddings()
    vect = Chroma(embedding_function=emb, persist_directory=chroma_persist_dir, collection_name=collection_name)
    # UPDATED: Using k=15 as per main.ipynb optimization
    retrieved = vect.similarity_search(roman_query, k=15)

    pieces = []
    for i, d in enumerate(retrieved):
        pieces.append(f'[{i+1}] Category: {d.metadata.get("category", "N/A")}\n{d.page_content}')
    context = '\n\n'.join(pieces)

    # UPDATED: Improved prompt to match successful main.ipynb template
    rag_prompt = f'''You are a helpful assistant for FAST-NUCES queries.\nAnswer the user using ONLY the provided context.\nIf the answer is not present in the context, clearly say you don\'t have enough information.\nKeep the answer concise and accurate.\n\nQuestion: {roman_query}\n\nContext:\n{context}\n\nProvide a JSON object with keys "roman_urdu" and "arabic_urdu" containing the answer in Roman-Urdu and Urdu (Arabic script) respectively.'''

    resp = openai.chat.completions.create(
        model=model,
        messages=[{'role':'system','content':'You are a JSON-output generator.'},{'role':'user','content':rag_prompt}],
        temperature=0.1,
    )
    assistant_text = resp.choices[0].message.content
    try:
        start = assistant_text.find('{')
        end = assistant_text.rfind('}')
        return json.loads(assistant_text[start:end+1])
    except:
        return {'roman_urdu': assistant_text, 'arabic_urdu': assistant_text}

In [7]:
# 4) TTS: synthesize Urdu (Arabic script) using gTTS and play/save it
def synthesize_urdu_to_mp3(urdu_text, out_path='response_urdu.mp3', lang='ur'):
    t = gTTS(text=urdu_text, lang=lang)
    t.save(out_path)
    display(Audio(out_path, autoplay=False))
    return out_path

In [8]:
# --- TEST THE PIPELINE ---
audio_file = 'tuition_fee.ogg'

print("Transcribing...")
urdu_script = transcribe_audio(audio_file)
print('Urdu Script:', urdu_script)

print("\nRomanizing...")
roman_question = urdu_to_roman(urdu_script)
print('Roman-Urdu:', roman_question)

print("\nQuerying RAG...")
resp = rag_query_and_generate(roman_question)
print('Roman-Urdu Response:', resp.get('roman_urdu'))
print('Urdu-Script Response:', resp.get('arabic_urdu'))

print("\nSynthesizing Voice...")
if resp.get('arabic_urdu'):
    synthesize_urdu_to_mp3(resp.get('arabic_urdu'))

Transcribing...
Urdu Script: فاسٹ یونیورسٹی کی ٹیوشن فیس کتنی ہے؟

Romanizing...
Roman-Urdu: Fast University ki tuition fees kitni hai?

Querying RAG...


c:\Users\-\OneDrive\Desktop\FYP2_P1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\-\AppData\Local\Temp\ipykernel_9812\3616435887.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vect = Chroma(embedding_function=emb, persist_directory=chroma_persist_dir, collection_name=collection_name)


Roman-Urdu Response: FAST University ki tuition fees Rs. 11,000 per credit hour hai.
Urdu-Script Response: فاسٹ یونیورسٹی کی ٹیوشن فیس Rs. 11,000 فی کریڈٹ گھنٹہ ہے۔

Synthesizing Voice...


In [9]:
# --- BATCH TEST PIPELINE ---
import glob

input_dir = 'Test_cases_inputs'
output_dir = 'test_results'
summary_file = os.path.join(output_dir, 'summary.json')
results = []

ogg_files = glob.glob(os.path.join(input_dir, '*.ogg'))
print(f"Found {len(ogg_files)} files to process.")

for audio_file in ogg_files:
    print(f"\nProcessing: {audio_file}")
    try:
        # 1. Transcribe
        urdu_script = transcribe_audio(audio_file)

        # 2. Romanize
        roman_question = urdu_to_roman(urdu_script)

        # 3. Query RAG
        resp = rag_query_and_generate(roman_question)
        roman_urdu_response = resp.get('roman_urdu', '')
        arabic_urdu_response = resp.get('arabic_urdu', '')

        # 4. Synthesize Voice
        out_filename = f"response_{os.path.basename(audio_file).replace('.ogg', '.mp3')}"
        out_path = os.path.join(output_dir, out_filename)

        if arabic_urdu_response:
            synthesize_urdu_to_mp3(arabic_urdu_response, out_path=out_path)

        # Collect result
        results.append({
            'input_audio': audio_file,
            'input_roman': roman_question,
            'response_roman': roman_urdu_response,
            'response_audio': out_path
        })
        print(f"Success: {out_filename}")

    except Exception as e:
        print(f"Error processing {audio_file}: {e}")

# Save summary
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"\nBatch processing complete. Summary saved to {summary_file}")


Found 8 files to process.

Processing: Test_cases_inputs\15.ogg


Success: response_15.mp3

Processing: Test_cases_inputs\19.ogg


Success: response_19.mp3

Processing: Test_cases_inputs\22.ogg


Success: response_22.mp3

Processing: Test_cases_inputs\3.ogg


Success: response_3.mp3

Processing: Test_cases_inputs\4.ogg


Success: response_4.mp3

Processing: Test_cases_inputs\6.ogg


Success: response_6.mp3

Processing: Test_cases_inputs\7.ogg


Success: response_7.mp3

Processing: Test_cases_inputs\8.ogg


Success: response_8.mp3

Batch processing complete. Summary saved to test_results\summary.json


In [11]:

test_query = "PHD applicants ke liye minimum requirements kya hain?"

resp = rag_query_and_generate(test_query)
roman_urdu_response = resp.get('roman_urdu', '')
arabic_urdu_response = resp.get('arabic_urdu', '')

print("\nSynthesizing Voice...")
if arabic_urdu_response:
    synthesize_urdu_to_mp3(arabic_urdu_response, out_path='test_single_response.mp3')
    print("Voice synthesis complete!")



Synthesizing Voice...


NameError: name 'synthesize_urdu_to_mp3' is not defined

In [11]:
# --- TEST FULL PIPELINE WITH VOICE INPUT ---
# Take voice file as input at runtime
voice_input_file = input("Enter the voice file path (e.g., Test_cases_inputs/3.ogg): ").strip()

if not os.path.exists(voice_input_file):
    print(f"Error: File '{voice_input_file}' not found!")
else:
    print(f"Processing voice input: {voice_input_file}\n")
    
    try:
        # 1. Transcribe voice to Urdu script
        print("Step 1: Transcribing audio...")
        urdu_script = transcribe_audio(voice_input_file)
        print(f"Transcribed Urdu: {urdu_script}\n")
        
        # 2. Convert Urdu script to Roman-Urdu
        print("Step 2: Romanizing Urdu...")
        roman_question = urdu_to_roman(urdu_script)
        print(f"Roman-Urdu: {roman_question}\n")
        
        # 3. Query RAG and generate response
        print("Step 3: Querying RAG...")
        resp = rag_query_and_generate(roman_question)
        roman_urdu_response = resp.get('roman_urdu', '')
        arabic_urdu_response = resp.get('arabic_urdu', '')
        print(f"Roman-Urdu Response: {roman_urdu_response}\n")
        print(f"Urdu-Script Response: {arabic_urdu_response}\n")
        
        # 4. Synthesize response to voice
        print("Step 4: Synthesizing response voice...")
        if arabic_urdu_response:
            output_file = 'full_pipeline_response.mp3'
            synthesize_urdu_to_mp3(arabic_urdu_response, out_path=output_file)
            print(f"Response saved to: {output_file}")
        
        print("\nFull pipeline test completed successfully!")
        
    except Exception as e:
        print(f"Error processing voice input: {e}")


Error: File '' not found!


In [ ]:
# Test gTTS with Arabic script Urdu
# Test with different Urdu texts (Arabic script)
test_texts = [
    "ہیلو، یہ ایک ٹیسٹ ہے۔",
    "کیا آپ مدد چاہتے ہیں؟",
    "فاسٹ نؤس میں داخلہ کے لیے کیا ضروریات ہیں؟"
]

for i, text in enumerate(test_texts):
    output_file = f'gtts_arabic_test_{i+1}.mp3'
    print(f"Testing gTTS with Arabic script: {text}")
    synthesize_urdu_to_mp3(text, out_path=output_file)
    print(f"Saved to: {output_file}\n")

Testing gTTS with: ہیلو، یہ ایک ٹیسٹ ہے۔


Saved to: gtts_test_1.mp3

Testing gTTS with: کیا آپ مدد چاہتے ہیں؟


Saved to: gtts_test_2.mp3

Testing gTTS with: فاسٹ نؤس میں داخلہ کے لیے کیا ضروریات ہیں؟


Saved to: gtts_test_3.mp3



In [17]:
# Test gTTS with Roman-Urdu
# Test with different Roman-Urdu texts
roman_test_texts = [
    "Hello, yeh ek test hai.",
    "Kya aap madad chahte hain?",
    "fast nuces mein admission ke liye kya zarooriyat hain?"
]

for i, text in enumerate(roman_test_texts):
    output_file = f'gtts_roman_test_{i+1}.mp3'
    print(f"Testing gTTS with Roman-Urdu: {text}")
    synthesize_urdu_to_mp3(text, out_path=output_file)
    print(f"Saved to: {output_file}\n")

Testing gTTS with Roman-Urdu: Hello, yeh ek test hai.


Saved to: gtts_roman_test_1.mp3

Testing gTTS with Roman-Urdu: Kya aap madad chahte hain?


Saved to: gtts_roman_test_2.mp3

Testing gTTS with Roman-Urdu: fast nuces mein admission ke liye kya zarooriyat hain?


Saved to: gtts_roman_test_3.mp3

